In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [4]:
df_job = pd.read_csv('../data/job_title_des.csv')
df_resume = pd.read_csv('../data/Resume.csv')

In [ ]:
data_keywords = [
    # Core Roles & Job Titles
    "data scientist", "senior data scientist", "machine learning engineer", "applied scientist",
    "research scientist", "data analyst", "business analyst", "bi analyst", "analytics engineer",
    "data engineer", "etl developer", "big data engineer", "mlops engineer", "ml engineer",
    "ai engineer", "decision scientist", "quantitative analyst", "quant analyst", "statistician",
    "data science intern", "data engineer intern", "analytics intern",

    # Programming Languages
    "python", "r", "sql", "scala", "java", "julia", "c", "c++", "c#", "matlab", "javascript",
    "typescript", "sas", "stata",

    # Python Data & ML Ecosystem
    "numpy", "pandas", "scipy", "scikit-learn", "sklearn", "statsmodels", "tensorflow", "keras",
    "pytorch", "jax", "xgboost", "lightgbm", "catboost", "prophet", "fbprophet", "pymc", "pymc3",
    "pymc4", "pyspark", "sparklyr", "dask", "ray", "rapids", "cudf", "cuml",

    # R Ecosystem
    "tidyverse", "dplyr", "tidyr", "ggplot2", "data.table", "caret", "mlr", "shiny", "rmarkdown",

    # Data Visualization & BI Tools
    "matplotlib", "seaborn", "plotly", "bokeh", "altair", "ggplot", "tableau", "tableau server",
    "tableau prep", "power bi", "looker", "looker studio", "mode analytics", "qlik", "qlikview",
    "qlik sense", "superset", "metabase", "redash", "grafana", "kibana", "excel dashboards",
    "google data studio",

    # Databases & Query Engines
    "mysql", "postgresql", "postgres", "sql server", "oracle", "sqlite", "snowflake", "bigquery",
    "amazon redshift", "redshift", "azure synapse", "teradata", "vertica", "mariadb", "db2",
    "presto", "trino", "hive", "impala", "clickhouse", "elasticsearch", "mongodb", "cassandra",
    "redis", "dynamodb", "neo4j",

    # Big Data & Distributed Computing
    "hadoop", "hdfs", "mapreduce", "spark", "apache spark", "spark sql", "spark streaming",
    "kafka", "apache kafka", "flink", "apache flink", "storm", "kinesis", "beam", "apache beam",
    "airflow on spark", "emr", "databricks", "cloudera", "hortonworks",

    # Cloud Platforms & Managed Data Services
    "amazon web services", "aws", "google cloud platform", "gcp", "microsoft azure", "azure",
    "cloud computing", "serverless", "aws s3", "s3", "aws lambda", "aws glue", "aws athena",
    "aws emr", "aws sagemaker", "azure data lake", "azure databricks", "azure ml",
    "azure functions", "google bigquery", "google cloud storage", "gcs", "google cloud functions",
    "vertex ai", "google cloud composer", "gcp dataflow", "gcp dataproc",
    "snowflake on aws", "snowflake on azure", "snowflake on gcp",

    # Data Engineering, ETL, and Workflow Orchestration
    "etl", "elt", "data pipelines", "batch processing", "streaming pipelines", "real-time data",
    "cdc", "change data capture", "data warehouse", "data lake", "data lakehouse", "data modeling",
    "dimensional modeling", "star schema", "snowflake schema", "kimball", "inmon", "olap", "oltp",
    "data integration", "data ingestion", "data transformation", "data normalization",
    "denormalization", "data quality", "data governance", "data lineage",
    "master data management", "mdm", "apache airflow", "airflow", "luigi", "prefect", "dagster",
    "dbt", "dbt core", "dbt cloud", "fivetran", "stitch", "talend", "informatica", "ssis",
    "data factory",

    # Machine Learning Fundamentals & Techniques
    "supervised learning", "unsupervised learning", "reinforcement learning", "semi-supervised learning",
    "self-supervised learning", "classification", "regression", "clustering", "anomaly detection",
    "outlier detection", "recommendation systems", "recommender systems", "ranking",
    "time series forecasting", "forecasting", "dimensionality reduction", "feature selection",
    "feature extraction", "cross-validation", "hyperparameter tuning", "model selection",
    "ensemble methods", "bagging", "boosting", "stacking", "random forests", "gradient boosting",
    "k-nearest neighbors", "knn", "support vector machines", "svm", "logistic regression",
    "linear regression", "ridge regression", "lasso", "elastic net", "decision trees",
    "naive bayes", "k-means", "hierarchical clustering", "dbscan", "pca", "t-sne", "umap",
    "arima", "sarima", "var", "prophet", "survival analysis",

    # Deep Learning & Specialized ML
    "neural networks", "deep learning", "cnn", "convolutional neural networks", "rnn",
    "recurrent neural networks", "lstm", "gru", "transformers", "attention mechanisms", "gan",
    "generative adversarial networks", "autoencoders", "variational autoencoders", "vae",
    "sequence models", "sequence-to-sequence", "seq2seq", "encoder-decoder", "computer vision",
    "image classification", "object detection", "segmentation", "opencv", "yolo", "mask r-cnn",
    "nlp", "natural language processing", "text classification", "named entity recognition", "ner",
    "sentiment analysis", "topic modeling", "lda", "word embeddings", "word2vec", "glove",
    "bert", "gpt", "large language models", "llms", "speech recognition",
    "recommendation engines", "collaborative filtering", "content-based filtering", "bandits",
    "contextual bandits", "reinforcement learning",

    # Statistics, Mathematics & Experimentation
    "descriptive statistics", "inferential statistics", "probability", "probability distributions",
    "hypothesis testing", "a/b testing", "experimentation", "statistical modeling",
    "regression analysis", "anova", "nonparametric tests", "confidence intervals", "p-values",
    "bayesian statistics", "bayesian inference", "mcmc", "markov chain monte carlo",
    "monte carlo simulation", "maximum likelihood estimation", "mle", "sampling methods",
    "bootstrapping", "time series analysis", "stochastic processes", "linear algebra",
    "optimization", "convex optimization",

    # MLOps, Model Serving & Productionization
    "mlops", "model deployment", "model serving", "production machine learning", "model monitoring",
    "model lifecycle", "ml pipelines", "ml orchestration", "ml platforms", "ci/cd",
    "continuous integration", "continuous delivery", "docker", "containers", "kubernetes", "k8s",
    "kubeflow", "mlflow", "experiment tracking", "model registry", "feature store", "feast",
    "tecton", "pipeline automation", "batch scoring", "online scoring", "shadow deployment",
    "canary deployment", "model drift", "data drift", "model explainability",
    "explainable ai", "xai", "shap", "lime", "fairness", "responsible ai",

    # General Software Engineering & DevOps Skills
    "git", "github", "gitlab", "bitbucket", "version control", "branching", "pull requests",
    "code review", "unit testing", "integration testing", "pytest", "unittest",
    "test-driven development", "tdd", "software engineering", "clean code", "design patterns",
    "rest apis", "building apis", "flask", "django", "fastapi", "microservices",
    "command line", "linux", "unix", "bash", "shell scripting", "jenkins",
    "github actions", "circleci", "azure devops", "terraform", "infrastructure as code",
    "monitoring", "logging", "prometheus", "elk stack",

    # Data Cleaning, Wrangling & Feature Engineering
    "data cleaning", "data wrangling", "data preprocessing", "data munging",
    "missing data imputation", "outlier handling", "feature engineering", "feature scaling",
    "normalization", "standardization", "encoding categorical variables", "one-hot encoding",
    "label encoding", "target encoding", "feature importance", "text preprocessing",
    "tokenization", "stemming", "lemmatization", "stop word removal", "n-grams",
    "regular expressions", "regex",

    # Analytics, Business, and Domain Keywords
    "business intelligence", "descriptive analytics", "diagnostic analytics", "predictive analytics",
    "prescriptive analytics", "kpi", "kpis", "metrics", "dashboards", "reporting",
    "performance metrics", "customer analytics", "marketing analytics", "product analytics",
    "growth analytics", "financial analytics", "operations analytics", "risk analytics",
    "churn analysis", "customer segmentation", "cohort analysis", "funnel analysis",
    "pricing analytics", "supply chain analytics", "logistics optimization", "fraud detection",
    "credit risk", "marketing attribution", "experimentation platform", "stakeholder management",
    "data storytelling", "data-driven decision making",

    # Common Tools for Analysts
    "excel", "microsoft excel", "google sheets", "pivot tables", "vlookup", "index match",
    "excel macros", "vba", "jupyter", "jupyter notebook", "jupyterlab", "rstudio", "vs code",
    "spyder", "sql queries", "ad-hoc analysis", "exploratory data analysis", "eda",
    "reporting automation",

    # Soft Skills & Impact Phrases
    "communication skills", "strong communication", "presenting insights", "stakeholder communication",
    "cross-functional collaboration", "working with product managers", "working with engineers",
    "working with business stakeholders", "problem solving", "critical thinking",
    "analytical mindset", "attention to detail", "self-starter", "fast learner", "leadership",
    "mentoring", "leading projects", "project management", "agile", "scrum", "kanban",

    # Education & Certifications Keywords
    "bachelor's degree in computer science", "bachelor's in statistics",
    "bachelor's in mathematics", "bachelor's in engineering",
    "master's degree in data science", "master's in computer science",
    "master's in statistics", "phd", "coursera", "edx", "udacity", "datacamp", "kaggle",
    "aws certified machine learning", "aws certified data analytics",
    "google professional data engineer",
    "google professional machine learning engineer",
    "microsoft certified data scientist",
    "databricks certification"
]
len(data_keywords)

526

In [34]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english" ,
    # built‑in English stopwords
    # you can also customize token_pattern or analyzer if you want
    vocabulary=data_keywords
)
# vectorizer.fit(data_keywords)
X = vectorizer.transform(df_job["Job Description"])  # X is bag‑of‑words matrix
feature_names = vectorizer.get_feature_names_out()

ValueError: Duplicate term in vocabulary: 'azure synapse'

In [33]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 61768 stored elements and shape (2277, 548)>

In [23]:
df_job['Job Description'][0]

'We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\nJob Types: Full-time, Part-time\nSalary: ₹20,000.00 - ₹40,000.00 per month\nBenefits:\nFlexible schedule\nFood allowance\nSchedule:\nDay shift\nSupplemental Pay:\nJoining bonus\nOvertime pay\nExperience:\ntotal work: 1 year (Preferred)\nHousing rent subsidy:\nYes\nIndustry:\nSoftware Development\nWork Remotely:\nTemporarily due to COVID-19'

In [29]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 319337 stored elements and shape (2277, 16407)>